# Notebook 05: Modelling Dataset and Masking Setup

This notebook prepares the final modelling dataset for PM2.5 imputation.

The previous notebooks completed the exploratory and validation stages:

- Notebook 01 selected the final monitoring stations and study period.
- Notebook 02 analysed PM2.5 missingness patterns.
- Notebook 03 examined temporal PM2.5 patterns.
- Notebook 04 examined temperature, spatial relationships, nearest neighbours, inverse-distance weights and chronological train-validation-test splits.


This notebook now moves from EDA to modelling preparation.

The main aims are:

1. Load the final processed PM2.5-temperature panel.
2. Recreate the chronological train-validation-test split.
3. Preserve the original PM2.5 values before any artificial masking.
4. Create real-missingness indicators.
5. Prepare the structure needed for artificial masking.
6. Save a modelling-ready dataset for statistical, machine-learning and deep-learning notebooks.

A key rule in this notebook is:

Artificial masking must only be applied to originally observed PM2.5 values.

Real missing PM2.5 values cannot be used for evaluation because their true values are unknown.

## Step 1: Setup and file paths

Loads required libraries and defines all input and output file paths used in this notebook.

In [1]:
# ============================================================
# Notebook 05: Modelling Dataset and Masking Setup
# Step 1: Setup and file paths
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Display settings make notebook outputs easier to inspect.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# Main data root folder.
DATA_ROOT = Path.home() / "Desktop" / "DISSERTATION" / "Codes" / "data"

# Processed folder contains cleaned datasets created in earlier notebooks.
PROCESSED_DIR = DATA_ROOT / "processed"

# Modelling folder contains modelling-ready datasets, masking files and evaluation targets.
# It is kept separate from processed data because these outputs are specifically prepared
# for statistical, machine-learning and deep-learning models.
MODELLING_DIR = DATA_ROOT / "modelling"
MODELLING_DIR.mkdir(parents=True, exist_ok=True)

# Main processed inputs from earlier notebooks.
panel_path = PROCESSED_DIR / "selected_pm25_temperature_panel.csv"
metadata_path = PROCESSED_DIR / "selected_station_metadata.csv"

# Spatial neighbour outputs created in Notebook 04.
top3_neighbours_path = PROCESSED_DIR / "top3_nearest_neighbours.csv"
idw_weights_path = PROCESSED_DIR / "top3_idw_weights.csv"

print("Data root directory:", DATA_ROOT)
print("Processed input directory:", PROCESSED_DIR)
print("Modelling output directory:", MODELLING_DIR)

Data root directory: C:\Users\Msi\Desktop\DISSERTATION\Codes\data
Processed input directory: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\processed
Modelling output directory: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling


## Step 2: Load final processed inputs

This step loads the final selected PM2.5-temperature panel and station metadata.

The main panel is the long-format station-hour dataset. Each row represents one monitoring station at one hourly timestamp.

The metadata table provides station-level information such as latitude, longitude and site type.

The nearest-neighbour and inverse-distance weighting tables are loaded because they will later be used to construct spatial PM2.5 features.

In [2]:
# ============================================================
# Step 2: Load final processed inputs
# ============================================================

panel = pd.read_csv(panel_path)
metadata = pd.read_csv(metadata_path)
top3_neighbours = pd.read_csv(top3_neighbours_path)
idw_weights = pd.read_csv(idw_weights_path)

# Datetime conversion is essential because later feature engineering depends on
# year, month, hour, day of week and chronological splitting.
panel["Datetime"] = pd.to_datetime(panel["Datetime"])

print("Panel shape:", panel.shape)
print("Metadata shape:", metadata.shape)
print("Top 3 neighbours shape:", top3_neighbours.shape)
print("IDW weights shape:", idw_weights.shape)

display(panel.head())
display(metadata.head())
display(top3_neighbours.head())
display(idw_weights.head())

Panel shape: (219120, 5)
Metadata shape: (5, 7)
Top 3 neighbours shape: (15, 4)
IDW weights shape: (15, 6)


,Station,Datetime,RawYear,PM25,Temperature
0,London Bexley,2021-01-01 01:00:00,2021,39.552,-1.2
1,London Bexley,2021-01-01 02:00:00,2021,52.359,-1.9
2,London Bexley,2021-01-01 03:00:00,2021,20.920,-2.3
3,London Bexley,2021-01-01 04:00:00,2021,15.943,-2.8
4,London Bexley,2021-01-01 05:00:00,2021,17.076,-3.2


,Station,Latitude,Longitude,SiteType,Zone,Agglomeration,LocalAuthority
0,London Bexley,51.466030,0.184806,Suburban Background,Greater London,Greater London Urban Area,Bexley
1,London Harlington,51.488790,-0.441614,Urban Industrial,Greater London,Greater London Urban Area,Hillingdon
2,London Honor Oak Park,51.449674,-0.037418,Urban Background,Greater London,Greater London Urban Area,Lewisham
3,London Marylebone Road,51.522530,-0.154611,Urban Traffic,Greater London,Greater London Urban Area,Westminster
4,London N. Kensington,51.521050,-0.213419,Urban Background,Greater London,Greater London Urban Area,Kensington


,Target_station,Neighbour_rank,Neighbour_station,Distance_km
0,London Bexley,1,London Honor Oak Park,15.50
1,London Bexley,2,London Marylebone Road,24.32
2,London Bexley,3,London N. Kensington,28.24
3,London Harlington,1,London N. Kensington,16.20
4,London Harlington,2,London Marylebone Road,20.22


,Target_station,Neighbour_rank,Neighbour_station,Distance_km,Inverse_distance,IDW_weight
0,London Bexley,1,London Honor Oak Park,15.50,0.064516,0.457
1,London Bexley,2,London Marylebone Road,24.32,0.041118,0.292
2,London Bexley,3,London N. Kensington,28.24,0.035411,0.251
3,London Harlington,1,London N. Kensington,16.20,0.061728,0.421
4,London Harlington,2,London Marylebone Road,20.22,0.049456,0.338


## Step 3: Basic input validation

Before creating the modelling dataset, this step checks whether the required input columns are present in the processed files.

Here, the word "validation" does not refer to the validation dataset used for model tuning. Instead, it means validating the structure of the input data.

This step checks that the main panel contains the essential columns needed for modelling:

- `Datetime`: the hourly timestamp
- `Station`: the monitoring station name
- `PM25`: the target variable to be imputed
- `Temperature`: the auxiliary meteorological predictor
- `RawYear`: the year used to create the chronological train-validation-test split

It also checks that the metadata table contains the station-level information needed for spatial modelling:

- `Station`
- `Latitude`
- `Longitude`
- `SiteType`

If any required column is missing, the notebook stops immediately using an assertion error. This prevents silent mistakes later in the workflow.

In [3]:
# ============================================================
# Step 3: Basic input validation
# ============================================================
#
# In this step, "validation" means checking that the input files
# have the correct structure before we create the modelling dataset.
#
# This is NOT the same as the validation split used later for model tuning.
#
# The aim is to make sure that all required columns are present.
# If an important column is missing, the notebook should stop here
# instead of producing incorrect results later.

# These are the minimum columns required from the main PM2.5-temperature panel.
# Datetime is needed for temporal features.
# Station is needed to identify monitoring sites.
# PM25 is the target variable for imputation.
# Temperature is the auxiliary meteorological predictor.
# RawYear is used later to create the chronological train-validation-test split.
required_panel_columns = {
    "Datetime",
    "Station",
    "PM25",
    "Temperature",
    "RawYear"
}

missing_panel_columns = required_panel_columns - set(panel.columns)

assert len(missing_panel_columns) == 0, (
    f"The panel is missing required columns: {missing_panel_columns}"
)

required_metadata_columns = {
    "Station",
    "Latitude",
    "Longitude",
    "SiteType"
}

missing_metadata_columns = required_metadata_columns - set(metadata.columns)

assert len(missing_metadata_columns) == 0, (
    f"The metadata table is missing required columns: {missing_metadata_columns}"
)

print("Input validation passed.")
print("Panel columns:")
print(panel.columns.tolist())

Input validation passed.
Panel columns:
['Station', 'Datetime', 'RawYear', 'PM25', 'Temperature']


## Step 4: Create the base modelling dataset

This step creates the main modelling dataframe.

The original PM2.5 values are preserved in a new column called `PM25_original`.

Later artificial masking will temporarily hide some observed PM2.5 values, but the original values must be kept as ground truth for evaluation.

The notebook also creates `is_real_missing`, which marks PM2.5 values that were already missing in the real monitoring data.

These real missing values are not evaluation targets.

In [4]:
# ============================================================
# Step 4: Create base modelling dataset
# ============================================================

model_df = panel.copy()

# Preserve the original PM2.5 values before any artificial masking.
# This column will later be used as the ground truth when evaluating model predictions.
model_df["PM25_original"] = model_df["PM25"]

# Real missingness means the PM2.5 value was already missing in the original dataset.
# These rows do not have known ground truth and should not be used as evaluation targets.
model_df["is_real_missing"] = model_df["PM25_original"].isna()

# Drop the raw PM25 column after copying it into PM25_original.
# From this point onwards, PM2.5 should be handled through:
# - PM25_original: the true original PM2.5 value
# - PM25_model_input: the value visible to models after artificial masking
#
# This reduces the risk of accidentally using the original target value as a model feature.
model_df = model_df.drop(columns=["PM25"])

print("Base modelling dataset shape:", model_df.shape)
print("Real missing PM2.5 values:", model_df["is_real_missing"].sum())
print("Observed PM2.5 values:", (~model_df["is_real_missing"]).sum())

display(model_df.head())

Base modelling dataset shape: (219120, 6)
Real missing PM2.5 values: 5107
Observed PM2.5 values: 214013


,Station,Datetime,RawYear,Temperature,PM25_original,is_real_missing
0,London Bexley,2021-01-01 01:00:00,2021,-1.2,39.552,False
1,London Bexley,2021-01-01 02:00:00,2021,-1.9,52.359,False
2,London Bexley,2021-01-01 03:00:00,2021,-2.3,20.920,False
3,London Bexley,2021-01-01 04:00:00,2021,-2.8,15.943,False
4,London Bexley,2021-01-01 05:00:00,2021,-3.2,17.076,False


## Step 5: Add station metadata

Station metadata is merged into the modelling dataset.

Latitude and longitude can be useful for spatial modelling, while site type can help models distinguish between different monitoring environments, such as urban background and urban traffic sites.

The merge is checked carefully to ensure that every station-hour row receives metadata.

In [5]:
# ============================================================
# Step 5: Add station metadata
# ============================================================

model_df = model_df.merge(
    metadata,
    on="Station",
    how="left"
)

missing_metadata_rows = model_df["Latitude"].isna().sum()

print("Shape after metadata merge:", model_df.shape)
print("Rows with missing metadata:", missing_metadata_rows)

assert missing_metadata_rows == 0, "Some rows did not receive station metadata."

display(model_df.head())

Shape after metadata merge: (219120, 12)
Rows with missing metadata: 0


,Station,Datetime,RawYear,Temperature,PM25_original,is_real_missing,Latitude,Longitude,SiteType,Zone,Agglomeration,LocalAuthority
0,London Bexley,2021-01-01 01:00:00,2021,-1.2,39.552,False,51.46603,0.184806,Suburban Background,Greater London,Greater London Urban Area,Bexley
1,London Bexley,2021-01-01 02:00:00,2021,-1.9,52.359,False,51.46603,0.184806,Suburban Background,Greater London,Greater London Urban Area,Bexley
2,London Bexley,2021-01-01 03:00:00,2021,-2.3,20.920,False,51.46603,0.184806,Suburban Background,Greater London,Greater London Urban Area,Bexley
3,London Bexley,2021-01-01 04:00:00,2021,-2.8,15.943,False,51.46603,0.184806,Suburban Background,Greater London,Greater London Urban Area,Bexley
4,London Bexley,2021-01-01 05:00:00,2021,-3.2,17.076,False,51.46603,0.184806,Suburban Background,Greater London,Greater London Urban Area,Bexley


## Step 6: Add temporal features

Temporal features are added because PM2.5 is strongly time-dependent.

These features help models learn regular patterns such as hourly, weekly and seasonal variation.

The basic temporal features created here are:

- month
- hour
- day of week
- weekend indicator
- season

`Year` is intentionally not used as a modelling feature because the chronological split is based on `RawYear`. Deriving a calendar year from `Datetime` could disagree with `RawYear` at year-boundary hours due to the hour-ending timestamp convention, so it is excluded to avoid inconsistency between the split logic and model features.

These features are simple, interpretable and useful for statistical and machine-learning models.


In [6]:
# ============================================================
# Step 6: Add temporal features
# ============================================================
#
# Temporal features are derived from the hourly Datetime column.
#
# Important:
# We intentionally do NOT create a separate Year feature from Datetime.
# The chronological split is based on RawYear, which follows the original
# AURN monitoring-year structure.
#
# Around year boundaries, Datetime.dt.year may not always perfectly match
# RawYear because of UK-AIR hour-ending timestamp conventions.
#
# Therefore:
# - RawYear is kept for split creation, audit and traceability.
# - Year is not used as a modelling feature.
# - Models will use Month, Hour, DayOfWeek, IsWeekend and Season instead.

model_df["Month"] = model_df["Datetime"].dt.month
model_df["Hour"] = model_df["Datetime"].dt.hour
model_df["DayOfWeek"] = model_df["Datetime"].dt.dayofweek

# Monday = 0, Sunday = 6.
# Saturday and Sunday are marked as weekend days.
model_df["IsWeekend"] = model_df["DayOfWeek"].isin([5, 6]).astype(int)

def assign_season(month):
    """
    Convert month number into meteorological season.

    Winter: December, January, February
    Spring: March, April, May
    Summer: June, July, August
    Autumn: September, October, November
    """
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

model_df["Season"] = model_df["Month"].apply(assign_season)

display(
    model_df[
        ["Datetime", "RawYear", "Station", "PM25_original", "Temperature", 
         "Month", "Hour", "DayOfWeek", "IsWeekend", "Season"]
    ].head()
)

,Datetime,RawYear,Station,PM25_original,Temperature,Month,Hour,DayOfWeek,IsWeekend,Season
0,2021-01-01 01:00:00,2021,London Bexley,39.552,-1.2,1,1,4,0,Winter
1,2021-01-01 02:00:00,2021,London Bexley,52.359,-1.9,1,2,4,0,Winter
2,2021-01-01 03:00:00,2021,London Bexley,20.920,-2.3,1,3,4,0,Winter
3,2021-01-01 04:00:00,2021,London Bexley,15.943,-2.8,1,4,4,0,Winter
4,2021-01-01 05:00:00,2021,London Bexley,17.076,-3.2,1,5,4,0,Winter


## Step 7: Recreate chronological train-validation-test split

The dataset is split chronologically rather than randomly.

This is necessary because random splitting would allow future observations to influence model training, creating temporal leakage.

The planned split is:

- Train: 2021 to 2023
- Validation: 2024
- Test: 2025

The validation set is used for tuning and model selection.

The test set is reserved for final evaluation.

In [7]:
# ============================================================
# Step 7: Recreate chronological train-validation-test split
# ============================================================

def assign_split(raw_year):
    """
    Assign each row to a chronological modelling split.

    RawYear is used rather than Datetime year because the processed AURN files
    are organised by raw monitoring year and previous capture checks were based on RawYear.
    """
    if raw_year in [2021, 2022, 2023]:
        return "Train"
    elif raw_year == 2024:
        return "Validation"
    elif raw_year == 2025:
        return "Test"
    else:
        return "Outside split"

model_df["Split"] = model_df["RawYear"].apply(assign_split)

split_counts = (
    model_df
    .groupby("Split")
    .agg(
        Rows=("PM25_original", "size"),
        Stations=("Station", "nunique"),
        Start_datetime=("Datetime", "min"),
        End_datetime=("Datetime", "max")
    )
    .reset_index()
)


# Order splits chronologically for a tidy display.
split_counts["Split"] = pd.Categorical(
    split_counts["Split"],
    categories=["Train", "Validation", "Test"],
    ordered=True,
)
split_counts = split_counts.sort_values("Split").reset_index(drop=True)

display(split_counts)

assert set(model_df["Split"].unique()) == {"Train", "Validation", "Test"}, \
    "Unexpected split label found."

assert model_df.groupby("Split")["Station"].nunique().eq(5).all(), \
    "Not every split contains all five stations."

print("Chronological split validation passed.")

,Split,Rows,Stations,Start_datetime,End_datetime
0,Train,131400,5,2021-01-01 01:00:00,2024-01-01
1,Validation,43920,5,2024-01-01 01:00:00,2025-01-01
2,Test,43800,5,2025-01-01 01:00:00,2026-01-01


Chronological split validation passed.


## Step 8: Check PM2.5 availability by split

This step checks how many PM2.5 values are observed and missing in each modelling split.

Artificial masking can only be applied to originally observed PM2.5 values.

The capture percentage is calculated as:

`PM2.5 capture (%) = observed PM2.5 values / total rows × 100`

In [8]:
# ============================================================
# Step 8: PM2.5 availability by split
# ============================================================

split_availability = (
    model_df
    .groupby("Split")
    .agg(
        Rows=("PM25_original", "size"),
        PM25_valid=("PM25_original", "count"),
        PM25_missing=("is_real_missing", "sum")
    )
    .reset_index()
)

split_availability["PM25_capture_%"] = (
    split_availability["PM25_valid"] / split_availability["Rows"] * 100
).round(2)

# Arrange rows in modelling order.
split_order = ["Train", "Validation", "Test"]
split_availability["Split"] = pd.Categorical(
    split_availability["Split"],
    categories=split_order,
    ordered=True
)

split_availability = split_availability.sort_values("Split").reset_index(drop=True)

display(split_availability)

,Split,Rows,PM25_valid,PM25_missing,PM25_capture_%
0,Train,131400,127512,3888,97.04
1,Validation,43920,43320,600,98.63
2,Test,43800,43181,619,98.59


## Step 9: Create initial masking columns

This step creates the columns needed for artificial masking.

At this stage, no artificial masking is applied yet.

The columns are:

- `is_artificial_masked`: whether an observed PM2.5 value has been hidden for evaluation
- `PM25_model_input`: the PM2.5 column that models will see

Initially, `PM25_model_input` is identical to `PM25_original`.

Later, artificially masked positions will be set to missing in `PM25_model_input`, while the true values remain preserved in `PM25_original`.

In [9]:
# ============================================================
# Step 9: Create initial masking columns
# ============================================================

# No artificial masking has been applied yet.
model_df["is_artificial_masked"] = False

# This is the PM2.5 column that models will use as input.
# Later, artificially masked values will be replaced with NaN here.
model_df["PM25_model_input"] = model_df["PM25_original"]

# Check that no artificial masks exist yet.
print("Artificially masked values:", model_df["is_artificial_masked"].sum())

display(
    model_df[
        ["Datetime", "Station", "Split", "PM25_original", 
         "PM25_model_input", "is_real_missing", "is_artificial_masked"]
    ].head()
)

Artificially masked values: 0


,Datetime,Station,Split,PM25_original,PM25_model_input,is_real_missing,is_artificial_masked
0,2021-01-01 01:00:00,London Bexley,Train,39.552,39.552,False,False
1,2021-01-01 02:00:00,London Bexley,Train,52.359,52.359,False,False
2,2021-01-01 03:00:00,London Bexley,Train,20.920,20.920,False,False
3,2021-01-01 04:00:00,London Bexley,Train,15.943,15.943,False,False
4,2021-01-01 05:00:00,London Bexley,Train,17.076,17.076,False,False


## Step 10: Save base modelling dataset before artificial masking

This file is the clean base modelling dataset.

It contains original PM2.5 values, real missingness indicators, temporal features, station metadata and chronological split labels.

Artificial masking has not been applied yet.

This file is useful because it provides a safe reusable starting point for different masking scenarios.

In [10]:
# ============================================================
# Step 10: Save base modelling dataset before artificial masking
# ============================================================

base_modelling_path = MODELLING_DIR / "base_modelling_dataset_before_masking.csv"

model_df.to_csv(base_modelling_path, index=False)

print("Saved base modelling dataset:")
print(base_modelling_path)
print("Shape:", model_df.shape)

Saved base modelling dataset:
C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\base_modelling_dataset_before_masking.csv
Shape: (219120, 20)


## Step 11: Define artificial masking strategy

This section defines the general artificial masking strategy used in this notebook.

The final masking scenarios are:

- `random10_point`
- `random30_point`
- `random50_point`
- `block24_10percent`
- `block72_10percent`

Artificial masking is applied only to originally observed PM2.5 values in the Validation and Test splits. The Train split is not artificially masked.

Real missing PM2.5 values are not used as evaluation targets because their true values are unknown.

Random point scenarios test missingness severity, while block-gap scenarios test the effect of structured missingness.


In [11]:
# ============================================================
# Step 11: Define artificial masking strategy
# ============================================================
#
# Proposal alignment:
# - Controlled masking uses a fixed random seed for reproducibility.
# - Artificial masking is applied only to originally observed PM2.5 values.
# - Models are later evaluated only at artificially masked positions.
# - Train is not artificially masked; Validation and Test are, because they
#   are used for model tuning and final evaluation.

# The three random point scenarios and their masking rates.
RANDOM_MASK_RATES = {
    "random10_point": 0.10,
    "random30_point": 0.30,
    "random50_point": 0.50,
}

RANDOM_SEED = 42
MASK_SPLITS = ["Validation", "Test"]

# Safety check:
# If the raw PM25 column is still present, drop it to reduce leakage risk.
# From this point onwards, PM2.5 is represented by:
# - PM25_original: true original value
# - PM25_model_input: value visible to models
if "PM25" in model_df.columns:
    model_df = model_df.drop(columns=["PM25"])
    print("Dropped raw PM25 column to reduce leakage risk.")
else:
    print("Raw PM25 column is already absent.")

print("Random masking rates:", RANDOM_MASK_RATES)
print("Random seed:", RANDOM_SEED)
print("Splits to mask:", MASK_SPLITS)
print("Train artificial masking: No")


Raw PM25 column is already absent.
Random masking rates: {'random10_point': 0.1, 'random30_point': 0.3, 'random50_point': 0.5}
Random seed: 42
Splits to mask: ['Validation', 'Test']
Train artificial masking: No


## Step 12: Create final random point masking scenarios

This step creates the final random point masking scenarios:

- `random10_point`
- `random30_point`
- `random50_point`

These scenarios test the effect of increasing missingness severity.

The masks are applied only to originally observed PM2.5 values in the Validation and Test splits. Train is not artificially masked.

A nested random design is used:

- the 10% mask is a subset of the 30% mask
- the 30% mask is a subset of the 50% mask

This makes the severity comparison cleaner because higher missingness scenarios extend the lower missingness scenario rather than using completely unrelated random positions.

In [12]:
# ============================================================
# Step 12: Create final random point masking scenarios
# ============================================================

def apply_random_point_mask_nested(
    df,
    mask_rate=0.10,
    mask_splits=("Validation", "Test"),
    seed=42,
    scenario_name="random10_point"
):
    """
    Apply random point artificial masking to originally observed PM2.5 values.

    This version uses a reproducible random permutation within each Split-Station
    group and then selects the first n rows from that permutation.

    Why this matters:
    If the same seed is used for 10%, 30% and 50%, the lower-rate masks become
    subsets of the higher-rate masks. This creates a cleaner severity comparison.

    Parameters
    ----------
    df : pandas.DataFrame
        Base modelling dataset.
    mask_rate : float
        Proportion of originally observed PM2.5 values to mask within each
        Split-Station group.
    mask_splits : tuple or list
        Splits where artificial masking should be applied.
    seed : int
        Random seed for reproducibility.
    scenario_name : str
        Name of the masking scenario.

    Returns
    -------
    masked_df : pandas.DataFrame
        Dataset with artificial masking applied.
    """

    masked_df = df.copy()

    # Reset masking columns so the function is safe to re-run.
    masked_df["PM25_model_input"] = masked_df["PM25_original"]
    masked_df["is_artificial_masked"] = False
    masked_df["MaskingScenario"] = scenario_name

    rng = np.random.default_rng(seed)
    selected_indices = []

    # Explicit sorting makes group order stable and reproducible.
    group_keys = (
        masked_df[["Split", "Station"]]
        .drop_duplicates()
        .sort_values(["Split", "Station"])
        .itertuples(index=False, name=None)
    )

    for split_name, station_name in group_keys:

        if split_name not in mask_splits:
            continue

        group = masked_df[
            (masked_df["Split"] == split_name) &
            (masked_df["Station"] == station_name)
        ]

        # Only originally observed PM2.5 values are eligible.
        eligible_index = group.loc[
            group["PM25_original"].notna() &
            (~group["is_real_missing"])
        ].index.to_numpy()

        n_to_mask = int(round(len(eligible_index) * mask_rate))

        if n_to_mask == 0:
            continue

        # A permutation makes 10%, 30% and 50% nested when the same seed is used.
        shuffled_index = rng.permutation(eligible_index)
        chosen_index = shuffled_index[:n_to_mask]

        selected_indices.extend(chosen_index.tolist())

    selected_indices = np.array(selected_indices)

    if len(selected_indices) > 0:
        masked_df.loc[selected_indices, "is_artificial_masked"] = True
        masked_df.loc[selected_indices, "PM25_model_input"] = np.nan

    return masked_df


masked_random10_df = apply_random_point_mask_nested(
    df=model_df,
    mask_rate=0.10,
    mask_splits=MASK_SPLITS,
    seed=RANDOM_SEED,
    scenario_name="random10_point"
)

masked_random30_df = apply_random_point_mask_nested(
    df=model_df,
    mask_rate=0.30,
    mask_splits=MASK_SPLITS,
    seed=RANDOM_SEED,
    scenario_name="random30_point"
)

masked_random50_df = apply_random_point_mask_nested(
    df=model_df,
    mask_rate=0.50,
    mask_splits=MASK_SPLITS,
    seed=RANDOM_SEED,
    scenario_name="random50_point"
)

print("random10_point masked rows:", int(masked_random10_df["is_artificial_masked"].sum()))
print("random30_point masked rows:", int(masked_random30_df["is_artificial_masked"].sum()))
print("random50_point masked rows:", int(masked_random50_df["is_artificial_masked"].sum()))

random10_point masked rows: 8650
random30_point masked rows: 25951
random50_point masked rows: 43250


## Step 13: Validate random point masking scenarios

This step checks that the random point masks follow the methodological rules:

- Train contains no artificial masks.
- Artificial masks appear only in Validation and Test.
- Only originally observed PM2.5 values are masked.
- Real missing PM2.5 values are not used as evaluation targets.
- Artificially masked values are hidden in `PM25_model_input`.
- Non-masked observed values remain visible in `PM25_model_input`.
- The random masks are nested: 10% is contained within 30%, and 30% is contained within 50%.

In [13]:
# ============================================================
# Step 13: Validate random point masking scenarios
# ============================================================

def validate_point_masking(masked_df, scenario_name, mask_splits):
    """
    Validate a random point masking scenario.
    """

    # Train must not be artificially masked.
    train_mask_count = masked_df.loc[
        masked_df["Split"] == "Train",
        "is_artificial_masked"
    ].sum()

    assert train_mask_count == 0, (
        f"{scenario_name}: Train contains {train_mask_count} artificial masks."
    )

    # Artificial masks must appear only in allowed splits.
    masked_splits_found = set(
        masked_df.loc[
            masked_df["is_artificial_masked"],
            "Split"
        ].unique()
    )

    assert masked_splits_found.issubset(set(mask_splits)), (
        f"{scenario_name}: masks found outside allowed splits: {masked_splits_found}"
    )

    # Artificially masked rows must have known original PM2.5 values.
    assert masked_df.loc[
        masked_df["is_artificial_masked"],
        "PM25_original"
    ].notna().all(), (
        f"{scenario_name}: some masked rows do not have PM25_original."
    )

    # Real missing rows must never be artificial evaluation targets.
    assert not masked_df.loc[
        masked_df["is_artificial_masked"],
        "is_real_missing"
    ].any(), (
        f"{scenario_name}: real missing PM2.5 values were masked."
    )

    # Artificially masked rows must be hidden from the model input.
    assert masked_df.loc[
        masked_df["is_artificial_masked"],
        "PM25_model_input"
    ].isna().all(), (
        f"{scenario_name}: some masked rows are still visible to models."
    )

    # Non-masked observed rows must remain visible to models.
    non_masked_observed = (
        masked_df["PM25_original"].notna() &
        (~masked_df["is_artificial_masked"])
    )

    assert masked_df.loc[
        non_masked_observed,
        "PM25_model_input"
    ].notna().all(), (
        f"{scenario_name}: some non-masked observed values became NaN."
    )

    # Real missing values should remain missing in model input.
    assert masked_df.loc[
        masked_df["is_real_missing"],
        "PM25_model_input"
    ].isna().all(), (
        f"{scenario_name}: some real missing values are visible in model input."
    )

    print(f"{scenario_name}: point masking validation passed.")


validate_point_masking(masked_random10_df, "random10_point", MASK_SPLITS)
validate_point_masking(masked_random30_df, "random30_point", MASK_SPLITS)
validate_point_masking(masked_random50_df, "random50_point", MASK_SPLITS)


# Nested mask check.
random10_indices = set(masked_random10_df.index[masked_random10_df["is_artificial_masked"]])
random30_indices = set(masked_random30_df.index[masked_random30_df["is_artificial_masked"]])
random50_indices = set(masked_random50_df.index[masked_random50_df["is_artificial_masked"]])

assert random10_indices.issubset(random30_indices), "random10 is not a subset of random30."
assert random30_indices.issubset(random50_indices), "random30 is not a subset of random50."

print("Nested random masking check passed: random10 ⊂ random30 ⊂ random50.")

random10_point: point masking validation passed.
random30_point: point masking validation passed.
random50_point: point masking validation passed.
Nested random masking check passed: random10 ⊂ random30 ⊂ random50.


## Step 14: Create helper function for saving scenario outputs

Each masking scenario should be saved consistently.

For every scenario, this helper saves:

1. The full masked modelling dataset.
2. The evaluation target table containing only artificially masked rows.
3. A summary table by split.
4. A summary table by split and station.

The evaluation target table will be used later to calculate MAE, RMSE and R².

In [14]:
# ============================================================
# Step 14: Create helper function for saving scenario outputs
# ============================================================

evaluation_target_columns = [
    "MaskingScenario",
    "Datetime",
    "RawYear",
    "Split",
    "Station",
    "SiteType",
    "Latitude",
    "Longitude",
    "PM25_original",
    "PM25_model_input",
    "Temperature",
    "Month",
    "Hour",
    "DayOfWeek",
    "IsWeekend",
    "Season",
    "is_real_missing",
    "is_artificial_masked"
]

def create_mask_summary_tables(masked_df):
    """
    Create summary tables for a masking scenario.
    """

    summary_by_split = (
        masked_df
        .groupby("Split")
        .agg(
            Total_rows=("PM25_original", "size"),
            Observed_PM25=("PM25_original", "count"),
            Real_missing_PM25=("is_real_missing", "sum"),
            Artificially_masked=("is_artificial_masked", "sum")
        )
        .reset_index()
    )

    summary_by_split["Artificial_mask_rate_%"] = (
        summary_by_split["Artificially_masked"] /
        summary_by_split["Observed_PM25"] * 100
    ).round(2)

    summary_by_split_station = (
        masked_df
        .groupby(["Split", "Station"])
        .agg(
            Total_rows=("PM25_original", "size"),
            Observed_PM25=("PM25_original", "count"),
            Real_missing_PM25=("is_real_missing", "sum"),
            Artificially_masked=("is_artificial_masked", "sum")
        )
        .reset_index()
    )

    summary_by_split_station["Artificial_mask_rate_%"] = (
        summary_by_split_station["Artificially_masked"] /
        summary_by_split_station["Observed_PM25"] * 100
    ).round(2)

    split_order = ["Train", "Validation", "Test"]

    summary_by_split["Split"] = pd.Categorical(
        summary_by_split["Split"],
        categories=split_order,
        ordered=True
    )

    summary_by_split_station["Split"] = pd.Categorical(
        summary_by_split_station["Split"],
        categories=split_order,
        ordered=True
    )

    summary_by_split = (
        summary_by_split
        .sort_values("Split")
        .reset_index(drop=True)
    )

    summary_by_split_station = (
        summary_by_split_station
        .sort_values(["Split", "Station"])
        .reset_index(drop=True)
    )

    return summary_by_split, summary_by_split_station


def save_masking_scenario_outputs(masked_df, scenario_name):
    """
    Save all outputs for one masking scenario.

    Outputs:
    - full masked modelling dataset
    - evaluation target table
    - summary by split
    - summary by split and station
    """

    dataset_path = MODELLING_DIR / f"modelling_dataset_{scenario_name}_masked.csv"
    targets_path = MODELLING_DIR / f"evaluation_targets_{scenario_name}.csv"
    summary_split_path = MODELLING_DIR / f"mask_summary_{scenario_name}_by_split.csv"
    summary_station_path = MODELLING_DIR / f"mask_summary_{scenario_name}_by_split_station.csv"

    # Save full scenario dataset.
    masked_df.to_csv(dataset_path, index=False)

    # Save evaluation targets only.
    targets = masked_df.loc[
        masked_df["is_artificial_masked"]
    ].copy()

    cols = [
        col for col in evaluation_target_columns
        if col in targets.columns
    ]

    targets = targets[cols]
    targets.to_csv(targets_path, index=False)

    # Save summary tables.
    summary_by_split, summary_by_split_station = create_mask_summary_tables(masked_df)

    summary_by_split.to_csv(summary_split_path, index=False)
    summary_by_split_station.to_csv(summary_station_path, index=False)

    print(f"Saved scenario: {scenario_name}")
    print("  Dataset:", dataset_path)
    print("  Evaluation targets:", targets_path)
    print("  Target rows:", targets.shape[0])

    return {
        "Scenario": scenario_name,
        "DatasetPath": dataset_path,
        "EvaluationTargetPath": targets_path,
        "SummaryBySplitPath": summary_split_path,
        "SummaryBySplitStationPath": summary_station_path,
        "EvaluationTargetRows": targets.shape[0]
    }

## Step 15: Save random point masking scenarios

This step saves the three random point masking scenarios:

- `random10_point`
- `random30_point`
- `random50_point`

These files will be reused by the statistical, machine-learning and deep-learning modelling notebooks.

In [15]:
# ============================================================
# Step 15: Save random point masking scenarios
# ============================================================

scenario_records = []

scenario_records.append(
    save_masking_scenario_outputs(masked_random10_df, "random10_point")
)

scenario_records.append(
    save_masking_scenario_outputs(masked_random30_df, "random30_point")
)

scenario_records.append(
    save_masking_scenario_outputs(masked_random50_df, "random50_point")
)

Saved scenario: random10_point
  Dataset: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\modelling_dataset_random10_point_masked.csv
  Evaluation targets: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\evaluation_targets_random10_point.csv
  Target rows: 8650
Saved scenario: random30_point
  Dataset: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\modelling_dataset_random30_point_masked.csv
  Evaluation targets: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\evaluation_targets_random30_point.csv
  Target rows: 25951
Saved scenario: random50_point
  Dataset: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\modelling_dataset_random50_point_masked.csv
  Evaluation targets: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\evaluation_targets_random50_point.csv
  Target rows: 43250


## Step 16: Define block-gap masking strategy

Block-gap scenarios represent structured sensor outages.

Unlike random point masking, the masking unit is a continuous wall-clock time window.

The two block-gap scenarios are:

- `block24_10percent`: 24 consecutive hourly timestamps
- `block72_10percent`: 72 consecutive hourly timestamps

Design rules:

1. Only Validation and Test are artificially masked.
2. Train is not artificially masked.
3. Only originally observed PM2.5 values are eligible.
4. Candidate windows must be fully observed.
5. Blocks must not overlap.
6. Blocks must not be adjacent.
7. At least one unmasked hour must separate two blocks.
8. The target masking rate is approximately 10%, because block masking moves in 24h or 72h units.

This design keeps the comparison clean:

- `random10_point` tests scattered 10% missingness.
- `block24_10percent` tests one-day structured gaps.
- `block72_10percent` tests multi-day structured gaps.

In [16]:
# ============================================================
# Step 16: Define block-gap masking strategy
# ============================================================

BLOCK_MASK_TARGET_RATE = 0.10
BLOCK_SPLITS = ["Validation", "Test"]
BLOCK_SEED = 42

print("Block masking target rate:", BLOCK_MASK_TARGET_RATE)
print("Block masking splits:", BLOCK_SPLITS)
print("Block seed:", BLOCK_SEED)

Block masking target rate: 0.1
Block masking splits: ['Validation', 'Test']
Block seed: 42


## Step 17: Create block-gap masking function

This function creates clean, non-overlapping and non-adjacent block gaps.

The function works separately within each Split-Station group.

A candidate block is accepted only if:

- it contains exactly `block_hours` consecutive hourly timestamps
- every PM2.5 value inside the block was originally observed
- it does not overlap with an already selected block
- it does not touch an already selected block

This means that `block24_10percent` contains only 24-hour masked runs and `block72_10percent` contains only 72-hour masked runs.

In [17]:
# ============================================================
# Step 17: Create block-gap masking function
# ============================================================

def apply_block_gap_mask(
    df,
    block_hours,
    target_rate=0.10,
    mask_splits=("Validation", "Test"),
    seed=42,
    scenario_name="block24_10percent"
):
    """
    Apply clean, non-overlapping, non-adjacent block-gap masking.

    A block is defined as block_hours consecutive hourly timestamps
    within a Split-Station group.

    Only fully observed windows are eligible. This means every selected
    24h block masks exactly 24 originally observed PM2.5 values, and every
    selected 72h block masks exactly 72 originally observed PM2.5 values.

    Blocks are separated by at least one unmasked hour.
    """

    masked_df = df.copy()

    # Reset masking columns so the function is safe to re-run.
    masked_df["PM25_model_input"] = masked_df["PM25_original"]
    masked_df["is_artificial_masked"] = False
    masked_df["MaskingScenario"] = scenario_name

    rng = np.random.default_rng(seed)
    selected_indices = []

    group_keys = (
        masked_df[["Split", "Station"]]
        .drop_duplicates()
        .sort_values(["Split", "Station"])
        .itertuples(index=False, name=None)
    )

    for split_name, station_name in group_keys:

        if split_name not in mask_splits:
            continue

        group = masked_df[
            (masked_df["Split"] == split_name) &
            (masked_df["Station"] == station_name)
        ].sort_values("Datetime")

        group_index = group.index.to_numpy()
        datetimes = group["Datetime"].to_numpy()
        observed = group["PM25_original"].notna().to_numpy()

        n_rows = len(group)
        n_observed = int(observed.sum())
        target_observed = int(round(n_observed * target_rate))

        if target_observed == 0:
            continue

        # Confirm the group has a continuous hourly timeline.
        # This makes positional windows equivalent to wall-clock hourly windows.
        datetime_diff = group["Datetime"].diff().dropna()
        assert (datetime_diff == pd.Timedelta(hours=1)).all(), (
            f"{split_name}/{station_name}: Datetime is not continuous hourly."
        )

        # Candidate clean windows:
        # A start position is eligible only when the full block is observed.
        obs_cumsum = np.concatenate([[0], np.cumsum(observed.astype(int))])

        clean_starts = []

        for start in range(0, n_rows - block_hours + 1):
            observed_count_in_window = obs_cumsum[start + block_hours] - obs_cumsum[start]

            if observed_count_in_window == block_hours:
                clean_starts.append(start)

        clean_starts = np.array(clean_starts)

        if len(clean_starts) == 0:
            print(f"Warning: no clean {block_hours}h windows for {split_name}/{station_name}.")
            continue

        # Shuffle candidate starts reproducibly.
        rng.shuffle(clean_starts)

        occupied = np.zeros(n_rows, dtype=bool)
        masked_observed = 0

        for start in clean_starts:

            if masked_observed >= target_observed:
                break

            # Add one-hour buffer on both sides.
            # This prevents adjacent blocks from merging into longer runs.
            buffer_start = max(0, start - 1)
            buffer_end = min(n_rows, start + block_hours + 1)

            if occupied[buffer_start:buffer_end].any():
                continue

            window_positions = np.arange(start, start + block_hours)

            # Mark only the actual block as occupied.
            # The buffer is checked during placement but remains unmasked.
            occupied[start:start + block_hours] = True

            chosen_indices = group_index[window_positions]
            selected_indices.extend(chosen_indices.tolist())

            masked_observed += block_hours

    selected_indices = np.array(selected_indices)

    if len(selected_indices) > 0:
        masked_df.loc[selected_indices, "is_artificial_masked"] = True
        masked_df.loc[selected_indices, "PM25_model_input"] = np.nan

    return masked_df

## Step 18: Create 24-hour and 72-hour block-gap scenarios

This step creates the two structured block-gap scenarios:

- `block24_10percent`
- `block72_10percent`

Both are applied only to Validation and Test.

The target rate is approximately 10% because block masking uses whole 24-hour or 72-hour units.

In [18]:
# ============================================================
# Step 18: Create 24-hour and 72-hour block-gap scenarios
# ============================================================

masked_block24_df = apply_block_gap_mask(
    df=model_df,
    block_hours=24,
    target_rate=BLOCK_MASK_TARGET_RATE,
    mask_splits=BLOCK_SPLITS,
    seed=BLOCK_SEED,
    scenario_name="block24_10percent"
)

masked_block72_df = apply_block_gap_mask(
    df=model_df,
    block_hours=72,
    target_rate=BLOCK_MASK_TARGET_RATE,
    mask_splits=BLOCK_SPLITS,
    seed=BLOCK_SEED,
    scenario_name="block72_10percent"
)

print("block24_10percent masked rows:", int(masked_block24_df["is_artificial_masked"].sum()))
print("block72_10percent masked rows:", int(masked_block72_df["is_artificial_masked"].sum()))

block24_10percent masked rows: 8784
block72_10percent masked rows: 9144


## Step 19: Validate block-gap masking scenarios

This step checks that the block-gap scenarios follow the required design.

The checks confirm that:

- Train has no artificial masks.
- Masks appear only in Validation and Test.
- Only originally observed PM2.5 values are masked.
- Real missing values are not evaluation targets.
- Masked values are hidden from `PM25_model_input`.
- Every masked run is exactly 24 hours or 72 hours, depending on the scenario.
- Masked runs are not adjacent.

In [19]:
# ============================================================
# Step 19: Validate block-gap masking scenarios
# ============================================================

def validate_block_masking(masked_df, scenario_name, block_hours, mask_splits):
    """
    Validate a block-gap masking scenario.
    """

    # Train must not be artificially masked.
    train_mask_count = masked_df.loc[
        masked_df["Split"] == "Train",
        "is_artificial_masked"
    ].sum()

    assert train_mask_count == 0, (
        f"{scenario_name}: Train contains {train_mask_count} artificial masks."
    )

    # Masks must appear only in allowed splits.
    masked_splits_found = set(
        masked_df.loc[
            masked_df["is_artificial_masked"],
            "Split"
        ].unique()
    )

    assert masked_splits_found.issubset(set(mask_splits)), (
        f"{scenario_name}: masks found outside allowed splits: {masked_splits_found}"
    )

    # Masked rows must have known original PM2.5.
    assert masked_df.loc[
        masked_df["is_artificial_masked"],
        "PM25_original"
    ].notna().all(), (
        f"{scenario_name}: some masked rows do not have PM25_original."
    )

    # Real missing rows must not be masked.
    assert not masked_df.loc[
        masked_df["is_artificial_masked"],
        "is_real_missing"
    ].any(), (
        f"{scenario_name}: real missing PM2.5 values were masked."
    )

    # Masked rows must be hidden from model input.
    assert masked_df.loc[
        masked_df["is_artificial_masked"],
        "PM25_model_input"
    ].isna().all(), (
        f"{scenario_name}: some masked rows are still visible to models."
    )

    # Non-masked observed rows must remain visible.
    non_masked_observed = (
        masked_df["PM25_original"].notna() &
        (~masked_df["is_artificial_masked"])
    )

    assert masked_df.loc[
        non_masked_observed,
        "PM25_model_input"
    ].notna().all(), (
        f"{scenario_name}: some non-masked observed values became NaN."
    )

    # Validate exact block length and separation.
    for (split_name, station_name), group in masked_df.groupby(["Split", "Station"]):

        if split_name not in mask_splits:
            continue

        group_sorted = group.sort_values("Datetime").reset_index(drop=True)
        mask_array = group_sorted["is_artificial_masked"].to_numpy()
        time_array = group_sorted["Datetime"].to_numpy()

        i = 0
        n = len(group_sorted)

        while i < n:

            if not mask_array[i]:
                i += 1
                continue

            start = i

            while i < n and mask_array[i]:
                i += 1

            end = i
            run_length = end - start

            assert run_length == block_hours, (
                f"{scenario_name}: {split_name}/{station_name} has a masked run "
                f"of {run_length}h, expected {block_hours}h."
            )

            # Check hourly continuity inside the masked run.
            run_times = pd.Series(time_array[start:end])
            run_diffs = run_times.diff().dropna()

            assert (run_diffs == pd.Timedelta(hours=1)).all(), (
                f"{scenario_name}: {split_name}/{station_name} masked run is not hourly continuous."
            )

            # Check non-adjacency before the run.
            if start > 0:
                assert not mask_array[start - 1], (
                    f"{scenario_name}: adjacent masked block found before a run."
                )

            # Check non-adjacency after the run.
            if end < n:
                assert not mask_array[end], (
                    f"{scenario_name}: adjacent masked block found after a run."
                )

    print(f"{scenario_name}: block masking validation passed.")


validate_block_masking(
    masked_block24_df,
    scenario_name="block24_10percent",
    block_hours=24,
    mask_splits=BLOCK_SPLITS
)

validate_block_masking(
    masked_block72_df,
    scenario_name="block72_10percent",
    block_hours=72,
    mask_splits=BLOCK_SPLITS
)

block24_10percent: block masking validation passed.
block72_10percent: block masking validation passed.


## Step 20: Save block-gap masking scenarios

This step saves the two block-gap scenarios:

- `block24_10percent`
- `block72_10percent`

These files will be used later to test whether models perform worse when missing values occur in structured gaps rather than as scattered random points.

In [20]:
# ============================================================
# Step 20: Save block-gap masking scenarios
# ============================================================

scenario_records.append(
    save_masking_scenario_outputs(masked_block24_df, "block24_10percent")
)

scenario_records.append(
    save_masking_scenario_outputs(masked_block72_df, "block72_10percent")
)

Saved scenario: block24_10percent
  Dataset: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\modelling_dataset_block24_10percent_masked.csv
  Evaluation targets: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\evaluation_targets_block24_10percent.csv
  Target rows: 8784
Saved scenario: block72_10percent
  Dataset: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\modelling_dataset_block72_10percent_masked.csv
  Evaluation targets: C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\evaluation_targets_block72_10percent.csv
  Target rows: 9144


## Step 21: Create final masking scenario summary

This step creates one final summary table covering all five masking scenarios.

The five scenarios are:

- `random10_point`
- `random30_point`
- `random50_point`
- `block24_10percent`
- `block72_10percent`

The random point scenarios test missingness severity.

The block-gap scenarios test missingness structure.

In [21]:
# ============================================================
# Step 21: Create final masking scenario summary
# ============================================================

def summarise_scenario_for_manifest(masked_df, scenario_name, masking_type, target_rate, block_hours=None):
    """
    Create a compact scenario-level summary for the final manifest.
    """

    summary = (
        masked_df
        .groupby("Split")
        .agg(
            Observed_PM25=("PM25_original", "count"),
            Artificially_masked=("is_artificial_masked", "sum")
        )
        .reset_index()
    )

    summary["Artificial_mask_rate_%"] = (
        summary["Artificially_masked"] /
        summary["Observed_PM25"] * 100
    ).round(2)

    validation_rate = summary.loc[
        summary["Split"] == "Validation",
        "Artificial_mask_rate_%"
    ].iloc[0]

    test_rate = summary.loc[
        summary["Split"] == "Test",
        "Artificial_mask_rate_%"
    ].iloc[0]

    train_masked = summary.loc[
        summary["Split"] == "Train",
        "Artificially_masked"
    ].iloc[0]

    total_masked = int(summary["Artificially_masked"].sum())

    return {
        "Scenario": scenario_name,
        "MaskingType": masking_type,
        "TargetRate": target_rate,
        "BlockHours": block_hours,
        "TrainMaskedRows": int(train_masked),
        "ValidationRate_%": validation_rate,
        "TestRate_%": test_rate,
        "TotalArtificiallyMasked": total_masked
    }


final_scenario_summary = pd.DataFrame([
    summarise_scenario_for_manifest(
        masked_random10_df,
        "random10_point",
        "Random point",
        0.10
    ),
    summarise_scenario_for_manifest(
        masked_random30_df,
        "random30_point",
        "Random point",
        0.30
    ),
    summarise_scenario_for_manifest(
        masked_random50_df,
        "random50_point",
        "Random point",
        0.50
    ),
    summarise_scenario_for_manifest(
        masked_block24_df,
        "block24_10percent",
        "Block gap",
        0.10,
        block_hours=24
    ),
    summarise_scenario_for_manifest(
        masked_block72_df,
        "block72_10percent",
        "Block gap",
        0.10,
        block_hours=72
    )
])

scenario_paths_df = pd.DataFrame(scenario_records)

final_scenario_summary = final_scenario_summary.merge(
    scenario_paths_df,
    on="Scenario",
    how="left"
)

scenario_order = [
    "random10_point",
    "random30_point",
    "random50_point",
    "block24_10percent",
    "block72_10percent"
]

final_scenario_summary["Scenario"] = pd.Categorical(
    final_scenario_summary["Scenario"],
    categories=scenario_order,
    ordered=True
)

final_scenario_summary = (
    final_scenario_summary
    .sort_values("Scenario")
    .reset_index(drop=True)
)

display(final_scenario_summary)

final_summary_path = MODELLING_DIR / "final_masking_scenario_summary.csv"
final_scenario_summary.to_csv(final_summary_path, index=False)

print("Saved final masking scenario summary:")
print(final_summary_path)

,Scenario,MaskingType,TargetRate,BlockHours,TrainMaskedRows,ValidationRate_%,TestRate_%,TotalArtificiallyMasked,DatasetPath,EvaluationTargetPath,SummaryBySplitPath,SummaryBySplitStationPath,EvaluationTargetRows
0,random10_point,Random point,0.1,NaN,0,10.00,10.00,8650,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,8650
1,random30_point,Random point,0.3,NaN,0,30.00,30.00,25951,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,25951
2,random50_point,Random point,0.5,NaN,0,50.00,50.00,43250,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,43250
3,block24_10percent,Block gap,0.1,24.0,0,10.14,10.17,8784,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,8784
4,block72_10percent,Block gap,0.1,72.0,0,10.64,10.50,9144,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,C:\Users\Msi\Desktop\DISSERTATION\Codes\data\m...,9144


Saved final masking scenario summary:
C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\final_masking_scenario_summary.csv


## Step 22: Check temperature availability at artificial target rows

This diagnostic checks whether the artificially masked PM2.5 evaluation targets also have temperature available. The masking design is not changed, but this summary is useful for interpreting later machine-learning and deep-learning models that use temperature as an auxiliary predictor.

In [22]:
# ============================================================
# Additional diagnostic: Temperature availability at artificial target rows
# ============================================================

def summarise_target_temperature_availability(masked_df, scenario_name):
    targets = masked_df.loc[masked_df["is_artificial_masked"]].copy()
    
    return {
        "Scenario": scenario_name,
        "TargetRows": len(targets),
        "TargetRows_TemperatureObserved": int(targets["Temperature"].notna().sum()),
        "TargetRows_TemperatureMissing": int(targets["Temperature"].isna().sum()),
        "TargetRows_TemperatureCapture_%": round(targets["Temperature"].notna().mean() * 100, 2),
        "Validation_TargetRows_TemperatureMissing": int(
            targets.loc[targets["Split"] == "Validation", "Temperature"].isna().sum()
        ),
        "Test_TargetRows_TemperatureMissing": int(
            targets.loc[targets["Split"] == "Test", "Temperature"].isna().sum()
        )
    }


target_temperature_availability = pd.DataFrame([
    summarise_target_temperature_availability(masked_random10_df, "random10_point"),
    summarise_target_temperature_availability(masked_random30_df, "random30_point"),
    summarise_target_temperature_availability(masked_random50_df, "random50_point"),
    summarise_target_temperature_availability(masked_block24_df, "block24_10percent"),
    summarise_target_temperature_availability(masked_block72_df, "block72_10percent")
])

display(target_temperature_availability)

target_temperature_availability_path = (
    MODELLING_DIR / "evaluation_target_temperature_availability.csv"
)

target_temperature_availability.to_csv(
    target_temperature_availability_path,
    index=False
)

print("Saved target temperature availability summary:")
print(target_temperature_availability_path)

,Scenario,TargetRows,TargetRows_TemperatureObserved,TargetRows_TemperatureMissing,TargetRows_TemperatureCapture_%,Validation_TargetRows_TemperatureMissing,Test_TargetRows_TemperatureMissing
0,random10_point,8650,8504,146,98.31,54,92
1,random30_point,25951,25480,471,98.19,156,315
2,random50_point,43250,42442,808,98.13,281,527
3,block24_10percent,8784,8573,211,97.60,75,136
4,block72_10percent,9144,8982,162,98.23,120,42


Saved target temperature availability summary:
C:\Users\Msi\Desktop\DISSERTATION\Codes\data\modelling\evaluation_target_temperature_availability.csv


## Step 23: Final Notebook 05 conclusion

Notebook 05 created the modelling-ready dataset and controlled artificial masking scenarios for the PM2.5 imputation experiments.

The final masking scenarios are:

- random point missingness at 10%, 30% and 50%
- 24-hour structured block gaps at approximately 10%
- 72-hour structured block gaps at approximately 10%

All artificial masks were applied only to originally observed PM2.5 values in the Validation and Test splits. The Train split was not artificially masked.

The original PM2.5 values are preserved in `PM25_original`, while `PM25_model_input` contains the values visible to models after masking.

The saved masked datasets and evaluation target tables will be used consistently in the statistical, machine-learning and deep-learning imputation notebooks.